# 训练 LLaVA

在 [准备CC3M数据集和数据加载器.ipynb](准备CC3M数据集和数据加载器.ipynb) 中，我们为我们的 LLaVA 模型创建了 Dataset 和 Collator，已经能够便利地获得训练需要用到的数据批量。本节，我们讲按照 [准备CC3M数据集和数据加载器.ipynb](准备CC3M数据集和数据加载器.ipynb) 描述的方案进行模型训练。

训练方式：

- 视觉塔（CLIP ViT-L/14-336）冻结；
- LLM（Qwen1.5-4B-Chat）用 LoRA 训练；
- 视觉-文本投影层（`multi_modal_projector`，两层 MLP）全量训练。

In [ ]:
# pip install transformers==4.41.2 peft==0.11.1

In [ ]:
import os

# 模型和数据都在本地，关闭 Hub 访问：
# 避免 PEFT 在保存 checkpoint 时把相对路径 show_model/model001 当成 Hub repo id 去联网重试
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# DataLoader fork 前禁用 tokenizers 并行，消掉 fork 警告
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 避免 libgomp 对 OMP_NUM_THREADS 非法值的警告；可按 CPU 核数调整
os.environ["OMP_NUM_THREADS"] = "8"

## 加载 Dataset 和 Collator

In [ ]:
import torch

# 从独立的 llava_data 模块加载数据相关常量、Dataset 和 Collator
from transformers import LlavaProcessor

from llava_data import (
    DATA_ROOT,
    IMAGE_DIR,
    MODEL_DIR,
    SEED,
    LlavaPretrainDataset,
    TrainCollator,
)

print(f"数据集 json 地址：{DATA_ROOT}")
print(f"解压后的图片地址：{IMAGE_DIR}")
print(f"待训练自建 LLaVA 模型地址：{MODEL_DIR}")
print(f"随机数种子：{SEED}")

In [ ]:
processor = LlavaProcessor.from_pretrained(MODEL_DIR)

# 使用 Dataset 加载训练数据集和测试数据集
train_dataset = LlavaPretrainDataset(DATA_ROOT / "chat_train.json", IMAGE_DIR, processor)
val_dataset = LlavaPretrainDataset(DATA_ROOT / "chat_val.json", IMAGE_DIR, processor)

print(f"训练样本数：{len(train_dataset)} \t 验证样本数：{len(val_dataset)}")

## 加载模型并接入 LoRA

把我们搭建的 LLaVA 模型加载进显存中，并关闭 KV Cache（仅在推理时用于加速推理，训练时使用的是自回归逻辑，不需要 KV Cache）。

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import LlavaForConditionalGeneration

# 加载待训练 LLaVA 模型
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},          # 能够指定不同部分在不同卡上运行，如 {"vision_tower": 0, "language_model": 1}
)
# 关闭 KV Cache，训练阶段不需要使用
model.config.use_cache = False

### LoRA

LoRA 低秩适配是提出用于参数高效微调（PEFT）的方法。在（微调）训练过程中，LoRA 不直接修改原始权重矩阵 $W$，而是在旁边并联两个小矩阵 $A$ 和 $B$，让权重更新变成 $W + \Delta W = W + B \times A$。

通常情况下可以认为，通过形状为 $ B [emb\_dim, rank] \times A [rank, emb\_dim]$ 的矩阵乘法，去近似原本的 $\Delta W [emb\_dim, emb\_dim]$，大幅减少微调时的参数。LoRA 原论文证明了这样的方法，在选择适当的 r 的情况下能够接近甚至超越全量微调的性能。$r$ 就是 $A$ 和 $B$ 相乘时的**中间维度**，也就是“秩”。

`lora_alpha` 控制 LoRA 分支的输出在最终结果中占多大比重，会整体放大或缩小 LoRA 更新对原始输出的影响强度。PEFT 默认的缩放方式是：

$$
s = \frac{\text{lora\_alpha}}{r}
$$

最终的线性层的输出是 $Wx + s \cdot BAx$。在本配置中 `lora_alpha=8`，而 `r=32`，所以缩放系数 $s = 8/32 = 0.25$。

CLIP 原论文在预训练和下游迁移中都冻结视觉塔，所以这里要保证 **CLIP 视觉塔全程冻结**。PEFT 的 `target_modules=["q_proj", ...]` 是按模块名后缀匹配的，CLIP 的 attention 里也有 `q_proj/k_proj/v_proj`，不加限制就会把 LoRA 误注入视觉塔。

In [ ]:
# 配置 LoRA
lora_config = LoraConfig(
    r=32,
    lora_alpha=8,
    lora_dropout=0.05,  # 抑制过拟合
    bias="none",        # 偏置增加参数量且在 PEFT 中收益有限，不使用
    
    # 给谁注入 LoRA：只匹配 language_model 下的 attention，避免命中 CLIP 的 q_proj/k_proj/v_proj
    target_modules=r".*language_model.*[.](q_proj|k_proj|v_proj|o_proj)$",
    # 谁做全量训练：除了 LoRA 适配器之外，还希望把 对齐单元 设为完全可训练，并最终保存在 checkpoint 里
    modules_to_save=["multi_modal_projector"],
    
    # 这是什么模型：CAUSAL_LM 即 因果语言模型
    # 帮助 PEFT 在包装模型时做正确的架构适配，比如识别哪些层属于语言模型、如何处理前向逻辑等
    task_type="CAUSAL_LM",
)

# 对 LLaVA 模型应用 LoRA 配置
model = get_peft_model(model, lora_config)
# 打印可训练参数数量
model.print_trainable_parameters()
# 因为选择使用了梯度检查点进行训练以节省显存，所以需要输入 embedding 参与梯度计算
model.enable_input_require_grads()

# 统计可训练的张量数量，并检查是否冻结 CLIP 权重
trainable_names = [name for name, param in model.named_parameters() if param.requires_grad]
print("trainable tensors:", len(trainable_names))
print("vision_tower trainable:", any("vision_tower" in name for name in trainable_names))
print("projector trainable:", any("multi_modal_projector" in name for name in trainable_names))

#### 关于梯度检查点冲突

**梯度检查点** (`gradient_checkpointing=True`) 是为了在训练阶段节省显存的技术。它在**前向传播时不保存中间激活值**，而是在**反向传播时重新计算**这些激活值，以便计算梯度。而在 PEFT (LoRA) 中，模型绝大部分参数是**冻结的**（`requires_grad=False`），只有 LoRA 适配器层的参数需要训练。

在反向传播进行重计算时，**PyTorch 的自动微分机制要求计算图中至少有一个输入需要梯度（`requires_grad=True`）**，否则重计算路径会断开，梯度无法继续向前传播。

在纯 LoRA 训练中，模型最开始的嵌入层（embedding）是**冻结的**，它的输出默认**不带有梯度**。这导致梯度在试图穿过这些冻结层到达 LoRA 层时“无路可走”，训练会静默失败或报错。

`enable_input_require_grads()` 找到模型的输入嵌入层，并强制将其输出的 `requires_grad` 属性设置为 `True`，**嵌入层的输出就被标记为“需要梯度”**，即使嵌入层的权重**本身仍然是冻结**的。这就在计算图中人为地创造了一个梯度“起点”，使得反向传播的链条能够顺利接通，让梯度可以从损失函数一路回传。

### 跑一个小 batch 测试损失是否能正常计算

LlavaForConditionalGeneration 统一化了训练时和推理时的模型接口。当传入的数据中 labels 不为空时，会计算并返回 loss。

In [ ]:
collator = TrainCollator(processor.tokenizer.pad_token_id)

batch = collator([train_dataset[0], train_dataset[1]])
batch = {key: value.to("cuda:0") for key, value in batch.items()}
batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)    # 图片输入降低精度

model.eval()
with torch.no_grad():
    loss = model(**batch).loss
print("sanity loss:", loss.item())
model.train()

## 训练模型

### 实时记录训练日志

HuggingFace Trainer 默认只在内存里维护 `state.log_history`。为了能够实时看到训练时 loss 的变化以及训练的相关状态，这里加一个 callback，每次 `on_log` 时把 metrics 追加到 JSONL 文件，训练过程中可以直接在另一个终端用 `tail -f` 查看或直接打开文件查看，训练结束后也能读取做可视化。

日志文件保存为 `output_llava_cc3m_lora/train_log.jsonl`。每行是一条 JSON，训练日志包含 `loss`、`learning_rate`、`grad_norm`、`epoch`，验证日志包含 `eval_loss`。默认每 10 个 optimizer step 一条；想更密可以把 `logging_steps` 改小。

In [ ]:
import json
import os
from datetime import datetime

from transformers import TrainerCallback

LOG_PATH = "output_llava_cc3m_lora/train_log.jsonl"

class JsonlLoggerCallback(TrainerCallback):
    def __init__(self, log_path):
        self.log_path = log_path
        log_dir = os.path.dirname(log_path)
        if log_dir:
            os.makedirs(log_dir, exist_ok=True)

    def on_train_begin(self, args, state, control, **kwargs):
        # 每次训练开始清空旧日志；如果想追加，改成 "a"
        with open(self.log_path, "w", encoding="utf-8"):
            pass

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        record = {
            "time": datetime.now().isoformat(timespec="seconds"),
            "step": state.global_step,
            "epoch": state.epoch,
            **logs,
        }
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()

### 训练配置

使用 HuggingFace transformers 的 Trainer 训练。具体参数的作用在下方的注释中。

In [ ]:
from transformers import Trainer, TrainingArguments

# 初始化数据整理器
collator = TrainCollator(processor.tokenizer.pad_token_id)

training_args = TrainingArguments(
    # 输出与日志
    output_dir="output_llava_cc3m_lora",    # 保存训练相关信息的目录
    logging_steps=10,                       # 每 10 个优化步记录一次日志，如 loss lr grad_norm
    logging_first_step=True,                # 第一步也记录日志
    report_to="none",                       # 不接入外部记录器
    
    # Batch 与梯度累积
    # 梯度累积等效 batch：per_device_train_batch_size × gradient_accumulation_steps × num_devices
    # 这里使用单卡，等效训练 batch=4*4=16, 验证 batch=2*1=2
    per_device_train_batch_size=4,          # 训练时 DataLoader batch=4
    per_device_eval_batch_size=2,           # 评估时 DataLoader batch=2，多卡时 loss 按样本数加权平均
    gradient_accumulation_steps=4,          # 每卡一次优化步 step 前积累多少次 DataLoader 的前向
    
    # 学习率与调度
    learning_rate=2e-4,                     # 最大学习率
    lr_scheduler_type="cosine",             # 余弦衰减学习率
    warmup_ratio=0.03,                      # 线性 warmup，指定前 3% 的 step
    
    # 精度
    bf16=True,                              # 统一使用 bf16 做前向/反向
    fp16=False,                             # 显示关闭 fp16 避免冲突
    
    # 评估与保存
    save_strategy="epoch",                  # 每个 epoch 结束存一次 checkpoint
    eval_strategy="steps",                  # 按优化步定期验证，让 loss 曲线上的 eval 点更密
    eval_steps=100,                         # 每 100 个优化器 step 验证一次
    save_total_limit=2,                     # 最多保留两个 checkpoint
    
    # 数据加载
    remove_unused_columns=False,            # 不通过模型的 forward 签名过滤数据字段
    dataloader_num_workers=4,               # 4 个 worker 并行做图片读取、resize、tokenize.
    dataloader_pin_memory=True,             # 把 batch 放到锁页内存中，便于 GPU 直接访问
    
    # 梯度检查点
    gradient_checkpointing=True,            # 使用梯度检查点节省显存
    gradient_checkpointing_kwargs={"use_reentrant": False}, 
    # 使用 PyTorch 新的非重入式 checkpoint 实现，和 PEFT/LoRA 兼容性更好
    
    # 总 epoch 数量优化器和随机种子
    num_train_epochs=2,
    optim="adamw_torch",
    seed=SEED,
)

### 训练模型

In [ ]:
trainer = Trainer(
    model=model,                    # 训练模型
    args=training_args,             # 训练配置
    train_dataset=train_dataset,    # 训练数据集
    eval_dataset=val_dataset,       # 验证数据集
    data_collator=collator,         # 数据整理器
    callbacks=[JsonlLoggerCallback(LOG_PATH)],  # 日志保存
)

trainer.train()
trainer.save_state()
trainer.save_model(training_args.output_dir)
processor.save_pretrained(training_args.output_dir)

#### 训练代码解释

`train()` 负责跑和存 checkpoint，`save_state()` 存训练状态，`save_model()` 存最终 adapter 权重，`processor.save_pretrained()` 存输入处理规则。

1. `trainer.train()` 启动训练。根据现在的设定，训练结束后，`output_dir` 下有：

    ```text
    output_llava_cc3m_lora/
    ├── checkpoint-313/        # 第 1 个 epoch 结束
    │   ├── adapter_config.json
    │   ├── adapter_model.safetensors
    │   ├── optimizer.pt
    │   ├── scheduler.pt
    │   ├── trainer_state.json
    │   ├── rng_state.pth
    │   └── ...
    ├── checkpoint-626/        # 第 2 个 epoch 结束
    └── train_log.jsonl
    ```

    这些 checkpoint 是能够**断点续训**用的，包含模型权重、优化器状态、调度器状态、随机数状态和 `trainer_state.json`。

2. `trainer.save_state()` 把 **Trainer 的运行状态**写到 `output_dir` 根目录，主要生成：

    ```text
    output_llava_cc3m_lora/trainer_state.json
    ```
    内容包括：

    - `global_step`、`epoch`；
    - `log_history`（所有训练/评估日志）；
    - `best_metric`、`best_model_checkpoint`；
    - `max_steps`、`num_train_epochs` 等。

3. `trainer.save_model(training_args.output_dir)` 把**最终模型**保存到 `output_dir` 根目录。产出：

    ```text
    output_llava_cc3m_lora/
    ├── adapter_config.json          # LoRA 配置（r、alpha、target_modules、modules_to_save 等）
    ├── adapter_model.safetensors    # LoRA 权重 + multi_modal_projector 的完整权重
    └── README.md                    # 可选
    ```

    - **只保存可训练部分**：LoRA 的 `lora_A/B`，以及 `modules_to_save` 里的 `multi_modal_projector` 完整权重；
    - **不保存冻结的基座权重**：CLIP 视觉塔和 Qwen 原始权重仍在 `show_model/model001`，加载时需要“基座 + adapter”两步。

4. `processor.save_pretrained(training_args.output_dir)` 存 **processor** 的配置，包括：

    ```text
    output_llava_cc3m_lora/
    ├── preprocessor_config.json     # 图像预处理：resize、crop、normalize 参数
    ├── tokenizer_config.json        # tokenizer 配置、chat_template
    ├── special_tokens_map.json      # <image> 等特殊 token 的映射
    ├── tokenizer.json               # 快速 tokenizer 的词表
    └── ...
    ```

    它保存“怎么把图片和文本变成模型输入”的规则。

### 训练过程可视化

使用之前保存的配置，绘制 loss-steps 图像。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

# 优先使用 JsonlLoggerCallback 的 LOG_PATH；单独运行本单元时退回默认路径
log_path = Path(LOG_PATH) if "LOG_PATH" in globals() else Path("output_llava_cc3m_lora/train_log.jsonl")
if not log_path.exists():
    raise FileNotFoundError(f"未找到训练日志 {log_path}，请先运行上面的训练代码")

records = []
with open(log_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# 训练日志带 "loss"，验证日志带 "eval_loss"；最后一条汇总日志只有 "train_loss"
train_points = [
    (r["step"], float(r["loss"]))
    for r in records
    if isinstance(r.get("loss"), (int, float))
]
eval_points = [
    (r["step"], float(r["eval_loss"]))
    for r in records
    if isinstance(r.get("eval_loss"), (int, float))
]
if not train_points and not eval_points:
    raise ValueError(f"{log_path} 中没有 loss / eval_loss 记录")

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)

# 训练 loss
if train_points:
    steps, losses = zip(*train_points)
    ax.plot(steps, losses, color="#2d6ca2", linewidth=1.4, label="Training loss")

# 验证 loss：每个 epoch 采一次，用点和竖虚线标出评估位置
if eval_points:
    eval_steps, eval_losses = zip(*eval_points)
    ax.plot(
        eval_steps,
        eval_losses,
        color="#b23a3a",
        linewidth=1.4,
        marker="o",
        markersize=4,
        label="Validation loss",
    )
    for step in eval_steps:
        ax.axvline(
            step, color="#b23a3a", alpha=0.12, linewidth=0.8,
            linestyle="--", zorder=0,
        )
    best_step, best_loss = min(eval_points, key=lambda point: point[1])
    ax.annotate(
        f"best {best_loss:.3f} @ step {best_step}",
        xy=(best_step, best_loss),
        xytext=(-8, 12),
        ha="right",
        textcoords="offset points",
        fontsize=8,
        color="#b23a3a",
    )

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("LLaVA CC3M LoRA training")
ax.grid(alpha=0.2)
ax.legend(frameon=False)
plt.tight_layout()

# 同时保存图片，训练结束后可以直接打开查看
fig_path = log_path.with_name("train_loss.png")
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"日志：{log_path}")
if train_points:
    print(f"训练 loss：{train_points[0][1]:.4f} -> {train_points[-1][1]:.4f}（共 {train_points[-1][0]} steps）")
if eval_points:
    print("验证 loss：" + "，".join(f"step {step}: {loss:.4f}" for step, loss in eval_points))
print(f"图片已保存到：{fig_path}")

## 使用模型进行推理

`Trainer` 保存的是 LoRA adapter（其中也包含 `multi_modal_projector` 的全量权重）。训练完成的 `model` 本身就是带 adapter 的 `PeftModel`，adapter 默认激活，可以直接 `generate`。运行前手动将下方的 relaunch 变量置为 False。

In [ ]:
relaunch = True

可以选择性保存合并后的模型。对于 LoRA 来说，不合并更加节省存储，加载时也不会慢很多，一般情况下可以选择不合并。

In [ ]:
if not relaunch:
    import shutil
    
    # 训练完成的 model 就是带 adapter 的 PeftModel，adapter 默认激活，可以直接推理。
    # 不建议在 AutoDL 上直接保存完整合并模型：会额外写约 9GB。
    merged_model = model
    
    free_gb = shutil.disk_usage(".").free / 2**30
    print(f"free disk: {free_gb:.1f} GB")
    
    # 确认磁盘空间足够（建议 > 12GB），保存一份将 adapter 合并进权重的模型：
    # merged_model = model.merge_and_unload()
    # merged_model.save_pretrained("output_llava_cc3m_lora_merged")
    # processor.save_pretrained("output_llava_cc3m_lora_merged")
    # print("merged model saved")

如果 notebook 的 内核 重启过，不需要重新训练。用下面的代码加载带 adapter 的模型。

In [ ]:
if relaunch:
    # 重新导入宏变量
    from llava_data import (
        DATA_ROOT,
        IMAGE_DIR,
        MODEL_DIR,
    )
    import torch
    from peft import PeftModel
    from transformers import LlavaProcessor, LlavaForConditionalGeneration

    MODEL_DIR = "show_model/model001"
    processor = LlavaProcessor.from_pretrained("output_llava_cc3m_lora")
    base = LlavaForConditionalGeneration.from_pretrained(
        MODEL_DIR, torch_dtype=torch.bfloat16, device_map={"": 0}
    )
    merged_model = PeftModel.from_pretrained(base, "output_llava_cc3m_lora")

用验证集里的几张图，对比参考 caption 和模型输出。模型应该能够生成与图片相关的内容。

In [ ]:
import json

from IPython.display import display
from PIL import Image


with open(DATA_ROOT / "chat_val.json", encoding="utf-8") as f:
    val_records = json.load(f)


def generate_caption(model, processor, record, max_new_tokens=40):
    human = record["conversations"][0]["value"]
    if human.count("<image>") != 1:
        human = "<image>\n" + human.replace("<image>", "").strip()

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": human},
    ]
    prompt = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image = Image.open(IMAGE_DIR / record["image"]).convert("RGB")

    inputs = processor(text=prompt, images=image, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )

    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()


samples = val_records[:5]
inference_results = []

for record in samples:
    image = Image.open(IMAGE_DIR / record["image"]).convert("RGB")

    # 显示用缩略图，不改变送入模型的原图
    show_image = image.copy()
    show_image.thumbnail((400, 400))
    print("IMAGE:", record["image"])
    display(show_image)

    reference = record["conversations"][1]["value"]
    prediction = generate_caption(merged_model, processor, record)
    print("REF :", reference)
    print("PRED:", prediction)
    print("-" * 80)

    inference_results.append({
        "image": record["image"],
        "reference": reference,
        "prediction": prediction,
    })

print(f"共推理 {len(inference_results)} 条，已存入 inference_results。")

下面的代码能够将推理输出通过 pdf 保存。

In [ ]:
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image


def add_inference_page(pdf, image, reference, prediction, page, total, image_name):
    """把一条样本渲染成 A4 的一页：图片在上，参考描述和模型输出在下。"""
    fig = plt.figure(figsize=(8.27, 11.69))   # A4 纵向，单位英寸
    fig.patch.set_facecolor("white")

    fig.text(0.07, 0.955, f"Sample {page}/{total}", fontsize=15, weight="bold")
    fig.text(0.07, 0.925, image_name, fontsize=9, color="#555555")

    # 图片区域：保持原图宽高比，居中放置
    ax = fig.add_axes([0.07, 0.41, 0.86, 0.50])
    ax.imshow(image)
    ax.set_axis_off()

    # 文本区域：按参考描述的行数动态下移预测部分
    line_h = 0.017
    ref_lines = textwrap.wrap(reference, width=90) or [""]
    pred_lines = textwrap.wrap(prediction, width=90) or [""]

    fig.text(0.07, 0.365, "Reference caption", fontsize=11, weight="bold")
    fig.text(
        0.07, 0.340, "\n".join(ref_lines),
        fontsize=10, va="top", linespacing=1.35,
    )

    pred_y = max(0.340 - len(ref_lines) * line_h - 0.045, 0.15)
    fig.text(
        0.07, pred_y, "Model prediction",
        fontsize=11, weight="bold", color="#b23a3a",
    )
    fig.text(
        0.07, pred_y - 0.025, "\n".join(pred_lines),
        fontsize=10, va="top", linespacing=1.35,
    )

    fig.text(
        0.07, 0.035, "LLaVA CC3M LoRA - validation sample",
        fontsize=8, color="#777777",
    )
    pdf.savefig(fig)
    plt.close(fig)


if "inference_results" not in globals():
    raise RuntimeError("请先运行上面的推理单元，生成 inference_results")
if not inference_results:
    raise ValueError("inference_results 为空，没有可保存的样本")

pdf_path = Path("output_llava_cc3m_lora/inference_samples.pdf")
pdf_path.parent.mkdir(parents=True, exist_ok=True)

with PdfPages(pdf_path) as pdf:
    for page, result in enumerate(inference_results, 1):
        image = Image.open(IMAGE_DIR / result["image"]).convert("RGB")

        # PDF 里用原图；过大的图片限制到 1600px，避免 PDF 体积过大
        pdf_image = image.copy()
        pdf_image.thumbnail((1600, 1600))
        add_inference_page(
            pdf, pdf_image, result["reference"], result["prediction"],
            page, len(inference_results), result["image"],
        )

print(f"PDF 已保存到：{pdf_path}")

## 支持多卡训练和重新指定数据集

TODO: 可以通过 `train_llava.py` 和 `llava_generate.py` 进行模型训练。